In [ ]:
%matplotlib widget
import matplotlib
#matplotlib.use('TkAgg')
import requests
import pandas as pd
import numpy as np
import time
from datetime import datetime, timedelta
from pprint import pprint
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt
from matplotlib.widgets import Button
from matplotlib.animation import FuncAnimation
from mpl_toolkits.basemap import Basemap
from sqlalchemy import create_engine, types, TIMESTAMP
import psycopg2
import contextily as ctx
import geopandas as gpd
from shapely.geometry import Point
import math
import json
print(ctx.__version__)
print(gpd.__version__)


In [ ]:
#declaring time range for AFAD api
current_time = datetime.now()
time_range_days = 1
time_range_hours = 24

def recent_time_range (time_until=current_time, range_in_hours=24):
    starting_time = time_until - timedelta(hours=range_in_hours)
    current_time_formatted = time_until.strftime("%Y-%m-%d %H:%M:%S")
    starting_time_formatted = starting_time.strftime("%Y-%m-%d %H:%M:%S")
    time_range_for_api = 'start=' + starting_time_formatted + '&end=' + current_time_formatted
    return time_range_for_api
    
start_time = current_time - timedelta(hours=1)
time_range = recent_time_range(current_time, 50)
pprint(time_range)
#start_time = format()

#time_stamp_for_table = datetime.strftime(start_time, "%Y-%m-%d")
#pprint(time_stamp_for_table)

In [ ]:
#importing credentials
with open("credentials.json", "r") as local_cred_file:
    postgresql_credentials = json.load(local_cred_file)

username = postgresql_credentials["username"]
password = postgresql_credentials["password"]
host = postgresql_credentials["host"]
port = postgresql_credentials["port"]
schema = "earthquake_data_vis"

In [ ]:
# declaring City 
choosen_city = '*****' # choose a city from Turkey
universal_conversion = {'Ç':'C',
                        'Ğ':'G',
                        'İ':'I',
                        'Ö':'O',
                        'Ş':'S',
                        'Ü':'U',
                        ' ':''}
choosen_city = choosen_city.upper()

"""for i in universal_conversion.keys():
    pprint(i)
    choosen_city = choosen_city.replace(i,universal_conversion[i])
pprint(choosen_city)"""

choosen_city = ''.join([universal_conversion[i] if i in universal_conversion.keys() else i for i in choosen_city])

pprint(choosen_city)



In [ ]:
try:
    conn = psycopg2.connect(
        host=host,
        database='earthquake_data_vis',
        user=username,
        password=password,
        port=port
    )

    cur = conn.cursor()
    print('connected successfully')
    
    choosen_city_str = "'" + choosen_city + "'"
    query_for_city = 'select * from "City_Coordinates_TR" where city = {}'.format(choosen_city_str)
    cur.execute(query_for_city)

    specified_coordinates = cur.fetchone()

    cur.close()
    conn.close()

    pprint(specified_coordinates)

except Exception as e:
    pprint(e)


In [ ]:
#formatting corrdinates for AFAD api (radial filtering)

#list_latlong = df_city_coordinates.loc[choosen_city]
list_latlong = specified_coordinates
pprint(list_latlong)
#lat = round(list_latlong[0], 2)
#lon = round(list_latlong[1], 2)
lat = list_latlong[1]
lon = list_latlong[2]

def lat_lon_for_api_radial(lat, lon, minrad = 0, maxrad = 100000):
    str_lat = str(lat)
    str_lon = str(lon)
    str_min_rad = str(minrad)
    str_max_rad = str(maxrad)
    temp_dict = {'str_lat1': str_lat,
                 'str_lon1': str_lon,
                 'str_max_rad1' : str_max_rad,
                 'str_min_rad1' : str_min_rad}
    str_for_api = 'lat={str_lat1}&lon={str_lon1}&maxrad={str_max_rad1}&minrad={str_min_rad1}'.format(**temp_dict)

    return [str_for_api, str_lat, str_lon, str_min_rad, str_max_rad]


def lat_lon_for_api_rectangular(lat, lon, dist_lat = 0.5, dist_lon = 0.5):
    global min_lat
    global max_lat
    global min_lon
    global max_lon
    min_lat = str(lat-dist_lat)
    max_lat = str(lat+dist_lat)
    min_lon = str(lon-dist_lon)
    max_lon = str(lon+dist_lon)
    temp_dict = {'min_lat1': min_lat,
                 'max_lat1': max_lat,
                 'min_lon_1' : min_lon,
                 'max_lon_1' : max_lon}
    str_for_api = 'minlat={min_lat1}&maxlat={max_lat1}&minlon={min_lon_1}&maxlon={max_lon_1}'.format(**temp_dict)

    return [str_for_api, min_lat, max_lat, min_lon, max_lon]


geo_filter_radial = lat_lon_for_api_radial(lat, lon)[0]

geo_filter_rectangular = lat_lon_for_api_rectangular(lat, lon)[0]

pprint(geo_filter_radial)
pprint(geo_filter_rectangular)

In [ ]:
#declaring format
declaring_format = 'json'
declared_format = 'format=' + declaring_format

In [ ]:
#declaring mag

In [ ]:
# downloading data using API (AFAD) and normalization

base_url = "https://deprem.afad.gov.tr"
endpoint = "/apiv2/event/filter?"
filters = [geo_filter_rectangular, time_range, declared_format]

url_for_api = base_url + endpoint

for i in filters:
    url_for_api = url_for_api + i + '&'

url_for_api = url_for_api[:-1]

print(url_for_api)

webdata = requests.get(url_for_api)


recent_earthquakes_json = webdata.json()
pprint(recent_earthquakes_json)
pprint(type(recent_earthquakes_json))
df_normalized_recent_earthquakes = pd.json_normalize(recent_earthquakes_json)
#earthquakes = normalized_recent_earthquakes[['date_time', 'geojson.type', 'geojson.coordinates', 'mag', 'depth', 'earthquake_id', 'provider', 'rev', 'location_tz']]

#normalized_earthquake_list = pd.json_normalize(earthquake_list, "date_time", "depth", "earthquake_id","geojson", "mag", "provider","title")
#normalized_earthquake_list = pd.json_normalize(earthquake_list)

#stop

In [ ]:
# Upload recent earthquake data into local database

#from sqlalchemy import create_engine

time_stamp_for_table = datetime.strftime(start_time, "%Y-%m-%d")

table_name_upload = choosen_city + '_' + time_stamp_for_table

data_types_for_recent_eq = {}

url = f"postgresql://{username}:{password}@{host}:{port}/{schema}" #already defined variables in the credentials import cell
engine = create_engine(url)
df_normalized_recent_earthquakes.to_sql(name=table_name_upload, 
                           con=engine,
                           schema='earthquake_data_vis',
                           if_exists='replace',
                           dtype=data_types_for_recent_eq)



In [ ]:
#change dtype of Postgresql earthquakes table 

try:
    conn = psycopg2.connect(
        host=host,
        database='earthquake_data_vis',
        user=username,
        password=password,
        port=port
    )

    cur = conn.cursor()
    print('connected successfully')
    
    
    query_to_change_dtypes = 'ALTER TABLE earthquake_data_vis."{}" ' \
                             'ALTER COLUMN "date" TYPE timestamp ' \
                             'USING date::timestamp;'.format(table_name_upload)

    cur.execute(query_to_change_dtypes)
    conn.commit()

    for notice in conn.notices:
        pprint(notice)

    cur.close()
    conn.close()


except Exception as e:
    pprint(e)



In [ ]:
# Download recent earthquake data from local database

time_stamp_for_table = datetime.strftime(start_time, "%Y-%m-%d")
table_name_download = choosen_city + '_' + time_stamp_for_table 
#(YYYY-MM-DD format)

try:
    conn = psycopg2.connect(
        host=host,
        database='earthquake_data_vis',
        user=username,
        password=password,
        port=port
    )

    print('connected successfully')
    
    query_for_recent_earthquakes = 'select * from "{}"'.format(table_name_download)
    pprint(query_for_recent_earthquakes)
    
    df_recent_earthquakes = pd.read_sql(query_for_recent_earthquakes, conn)

    conn.close()

    pprint(df_recent_earthquakes)

except Exception as e:
    pprint(e)

df_recent_earthquakes.sort_values(by='date', ascending=True, inplace=True) # for animation to start with the earliest event


In [ ]:
# Create geodataframe, Adjust projection and define plotting boundries

# 1. Define coordinate in latitude/longitude (EPSG:4326)
lat, lon = list_latlong[1], list_latlong[2]

xmin = lat_lon_for_api_rectangular(lat, lon)[3]
xmax = lat_lon_for_api_rectangular(lat, lon)[4]
ymin = lat_lon_for_api_rectangular(lat, lon)[1]
ymax = lat_lon_for_api_rectangular(lat, lon)[2]

# 2. Create a GeoDataFrame
gdf_events = gpd.GeoDataFrame(
    df_recent_earthquakes,
    geometry=gpd.points_from_xy(df_recent_earthquakes.longitude, df_recent_earthquakes.latitude),
    crs='EPSG:4326'
)
#pprint(gdf_events)

gdf_for_limits = gpd.GeoDataFrame(
    {'location': ['Lower_Left_Lim','Upper_Right_Lim']},
    geometry=[Point(xmin, ymin), Point(xmax, ymax)],
    crs='EPSG:4326'
)
pprint(gdf_for_limits)

# Plot and set limits

pprint(ymin)
pprint(ymax)
pprint(xmin)
pprint(xmax)

# 3. Convert to Web Mercator (Required for contextily basemaps)
gdf_mercator = gdf_events.to_crs(epsg=3857)
gdf_for_limits_mercator = gdf_for_limits.to_crs(epsg=3857)

pprint(gdf_mercator)
pprint(gdf_for_limits_mercator)

xmin_converted = gdf_for_limits_mercator['geometry'].x[0]
ymin_converted = gdf_for_limits_mercator['geometry'].y[0]
xmax_converted = gdf_for_limits_mercator['geometry'].x[1]
ymax_converted = gdf_for_limits_mercator['geometry'].y[1]

pprint(xmin_converted)


In [ ]:
# 4. Create the plot and axis

# defining x/y proportion for actual plotting area

x_length = abs(xmax_converted - xmin_converted)
y_length = abs(ymax_converted - ymin_converted)

plot_ratio = y_length/x_length
plot_ratio = round(plot_ratio,1)
plot_x_lenght = 10 # inches
plot_y_length = plot_x_lenght * plot_ratio

fig, ax = plt.subplots(figsize=(plot_x_lenght, plot_y_length))
ax.set_xlim([xmin_converted, xmax_converted])
ax.set_ylim([ymin_converted, ymax_converted])
#plt.ion()
#plt.gca().set_aspect('equal')
plt.grid(True)

In [ ]:
# animation version 1

gdf_animation = gdf_mercator.copy()

x_coords = gdf_animation.geometry.x.to_numpy()
y_coords = gdf_animation.geometry.y.to_numpy()
num_points = len(gdf_animation)

scat = ax.scatter(
    x_coords,
    y_coords,
    color='mediumvioletred', 
    s=50, 
    edgecolor='white', 
    linewidth=0.7, 
    alpha=0.8
)
ctx.add_basemap(ax, source=ctx.providers.OpenStreetMap.Mapnik, zoom=10)
plt.figtext(
    0.5,
    0.01,
    '© OpenStreetMap contributors',
    ha='center',
    fontsize=8,
)

update_time_in_miliseconds = 300
dots_to_plot = []

def update(frame):
    ax.set_title(f"Simultaneous Markersize Animation - Frame {frame}", fontsize=14, fontweight='bold')
    
    # --- CALCULATE YOUR SIZES HERE ---
    # Example 1: A uniform breathing/pulsing effect for all points together
    # need to come up with better animation for point pulses. Must be dependent to dot_frame.
    base_pulse = 10 + np.sin(frame * 0.05) * 200
    base_pulse = abs(base_pulse)
    new_sizes = np.full(num_points, base_pulse)
    """pulse_timer = np.modf(frame * update_time_in_miliseconds / interval)
    base_pulse = pulse_timer * 100 
    new_sizes = np.full(num_points, base_pulse)"""
    
    # Example 2: Individual fluctuating sizes for each point independently
    # (e.g., simulating sensor readings, traffic, or data spikes)
    # Sizes in matplotlib scatter are proportional to the area (pixels squared)
    #new_sizes = 40 + np.sin(frame * 0.3 + np.arange(num_points)) * 30 + np.random.uniform(0, 10, num_points)
    # ----------------------------------
    
    # Inject the new size array directly into the existing canvas markers
    scat.set_sizes(new_sizes)
    
    new_dot_timer =  update_time_in_miliseconds / interval
    dot_frame = int(frame/new_dot_timer) #timer for dots to appear


    dots_to_plot.append([x_coords[dot_frame],y_coords[dot_frame]]) #coordinates will be appended here according to frame and this array will be choosed instead xcoords[],ycoords[]
                      # format will be [[x1,y1],[x2,y2]...]

    scat.set_offsets((dots_to_plot))
    
    return scat,

# 6. Run the live animation loop
# blit=True works perfectly here for ultra-smooth rendering
interval = 20
frame = int(num_points * update_time_in_miliseconds / interval)
anim = FuncAnimation(fig, update, frames=frame, interval=interval, blit=True, repeat=True)



In [ ]:
#adding buttons (with the new animator this functionality failed)

clean_button_place = plt.axes([0.8, 0.01, 0.1, 0.03]) #(left, bottom, width, height)
clean_button = Button(clean_button_place, 'Clean', color='tomato', hovercolor='red')

stop_button_place = plt.axes([0.8, 0.04, 0.1, 0.03]) #(left, bottom, width, height)
stop_button = Button(stop_button_place, 'Stop', color='yellow', hovercolor='orange')

restart_button_place = plt.axes([0.8, 0.07, 0.1, 0.03]) #(left, bottom, width, height)
restart_button = Button(restart_button_place, 'Restart', color='green', hovercolor='blue')

def cleaning_the_scatter(event):
    # 
    global scat 
    """
    if scat in ax.collections:
        scat.remove()
        print("Succesfully Refreshed!")
    else:
        print("No data to refresh!")
    """
    scat.set_visible(False)
    pprint(scat)

def stopping_the_animation(event):
    try:
        #anim.event_source.stop()
        anim.pause()
        print("Succesfully Stopped!")
    except:
        print("Failed to Stop!")

def restarting_the_animation(event):
    try:
        for text in ax.texts:
            text.remove()
 
        anim.frame_seq = anim.new_frame_seq()
        #anim.event_source.start()
        dots_to_plot.clear()
        
        anim.resume()
        scat.set_visible(True)
        #plt.show()
        print("Started!")
    except:
        print("Start operation failed!")


clean_button.on_clicked(cleaning_the_scatter)
stop_button.on_clicked(stopping_the_animation)
restart_button.on_clicked(restarting_the_animation)

plt.show()

In [ ]:
# 1. Stop the internal animation timer immediately
anim.event_source.stop()

if scat in ax.collections:
    scat.remove()
#fig.canvas.draw_idle()

# 2. Tell the widget canvas to completely close and destroy its UI element
#plt.close(fig) 